# Orbit smoothing proof of concept

This notebook walks through independent truth generation, past-trajectory estimation, and uncertainty checks. It starts with a two-hour development case; this is **not** the full calibrated Monte Carlo benchmark. The command-line stages below produce the complete experiment.

Run this notebook in the inference environment. Set `TUDAT_PYTHON` to the Python executable in the separate Tudat environment. No paid data is needed. Positions are meters, velocities are meters/second, and times are elapsed seconds.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from orbit_poc.benchmark import generate_one, fit_one, read_json, schur_check
from orbit_poc.data import load_dataset
from orbit_poc.report import make_report

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUN = PROJECT / 'runs' / 'notebook-demo'
TUDAT_PYTHON = os.environ.get('TUDAT_PYTHON')
config = dict(orbit='leo500', atmosphere='disturbed', coverage='normal',
              duration_hours=2.0, cadence=60.0, seed=42,
              epoch='2024-05-10T00:00:00', truth_profile='controlled',
              pulse_center_hours=1.0, pulse_width_hours=0.25, substep=2.5)
config

## Independent simulation

Tudat produces truth using independent numerical propagation. The controlled force model contains central gravity, J2, and drag in a co-rotating atmosphere. A smooth atmospheric disturbance is not sampled from the estimator's OU prior. `dataset.npz` contains only observations, initial-state priors, and scenario metadata. `truth.npz` is loaded separately for evaluation.

The SciPy backend is useful for debugging, but its results are explicitly excluded from independent-truth benchmark validity.

In [ ]:
generated = generate_one(config, RUN / 'data', truth_backend='tudat', tudat_python=TUDAT_PYTHON)
dataset = load_dataset(generated['dataset'])
print('Trajectory nodes:', len(dataset.times))
print('Range/range-rate observation pairs:', len(dataset.observations['node']))
generated['metadata']

## Compare identical data with different uncertainty models

The deterministic baseline estimates the initial orbit and one constant log-drag correction. Stochastic models use white empirical acceleration, OU-correlated empirical acceleration, or an OU log-density correction. A shared deterministic initializer and frozen process covariance define the objective for both GTSAM and IEKS.

These demonstration parameters are defaults. The final study selects parameters on independent calibration seeds using held-out predictive likelihood.

In [ ]:
results = {}
for model, solver in [('baseline', 'deterministic'), ('white', 'gtsam'), ('colored', 'gtsam'), ('density', 'gtsam')]:
    parameters = dict(model='white' if model == 'baseline' else model, substep=2.5)
    context = {**config, 'model': model, 'solver': solver}
    results[model] = fit_one(generated['dataset'], parameters, RUN / 'fits' / model,
                             solver=solver, options={'max_iterations': 20},
                             truth_path=generated['truth'], run_context=context)
[(name, fit['converged'], fit['metrics']['position_rmse_m']) for name, fit in results.items()]

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True, constrained_layout=True)
with np.load(RUN / 'fits' / 'density' / 'epoch_metrics.npz') as values:
    hours = (values['times'] - values['times'][0]) / 3600
    for axis, index, label in zip(axes, range(3), ['Radial', 'Along-track', 'Cross-track']):
        axis.plot(hours, values['rtn_errors'][:, index], label='Estimation error')
        bound = 1.96 * values['rtn_position_sigma'][:, index]
        axis.fill_between(hours, -bound, bound, alpha=0.2, label='Local Gaussian 95% interval')
        axis.set_ylabel(label + ' (m)')
axes[0].legend()
axes[-1].set_xlabel('Elapsed hours')
plt.show()

## Solver consistency and trajectory splitting

GTSAM and an iterated Kalman/RTS smoother should converge to the same local optimum for the same graph and frozen process noise. Compare objective values and states only after both converge. The separator test checks exact elimination for a supplied Gaussian linearization; it does not assert exact nonlinear marginalization.

In [ ]:
ieks = fit_one(generated['dataset'], {'model': 'density', 'substep': 2.5},
               RUN / 'fits' / 'density-ieks', solver='ieks',
               options={'max_iterations': 20}, truth_path=generated['truth'],
               run_context={**config, 'model': 'density', 'solver': 'ieks'})
print('Objectives:', results['density']['objective'], ieks['objective'])
print('Same frozen-Q reference:', results['density']['reference_sha256'] == ieks['reference_sha256'])
schur_check()

## Run the prespecified benchmark

Execute these commands from the project directory after the numerical verification gates pass. The pilot determines whether 5, 10, or 20 seeds fit the available runtime; it never labels two seeds as a final benchmark. Eight CPU workers each receive two CPUs and one BLAS thread. Independent per-orbit calibration processes can be merged with `merge-calibration`.

```bash
python -m orbit_poc.cli calibrate --config configs/scenarios.json --output-dir runs/calibration --max-iterations 60 --tudat-python "$TUDAT_PYTHON"
python -m orbit_poc.cli benchmark --config configs/scenarios.json --output-dir runs/pilot --calibration runs/calibration/calibration.json --pilot --workers 8 --max-iterations 60 --tudat-python "$TUDAT_PYTHON"
python -m orbit_poc.cli benchmark --config configs/scenarios.json --output-dir runs/final --calibration runs/calibration/calibration.json --seeds 20 --workers 8 --budget-hours 12 --max-iterations 60 --tudat-python "$TUDAT_PYTHON"
python -m orbit_poc.cli scaling --config configs/scenarios.json --output-dir runs/final/scaling --max-iterations 60 --tudat-python "$TUDAT_PYTHON"
python -m orbit_poc.cli posterior --config configs/scenarios.json --output-dir runs/final/posterior --max-iterations 60 --budget-hours 3 --job-timeout 1800 --tudat-python "$TUDAT_PYTHON"
python -m orbit_poc.cli report --input-dir runs/final
```

For a developmental smoke run, use `benchmark --pilot --max-jobs 1` or `calibrate --quick`. Their manifests and reports remain explicitly incomplete. The posterior stage tests the three models on quiet 500 km arcs, density on a disturbed 500 km arc with a 30-minute blackout, and density on a quiet 800 km arc. Four-chain NUTS must pass convergence diagnostics before its output is a posterior reference.

In [ ]:
report = make_report(RUN)
print(report['status'])
print(report['html'])

## Interpretation

- Use paired seeds to compare position/velocity RMSE and runtime. Do not count correlated time samples as independent Monte Carlo draws.
- Evaluate uncertainty coverage across independent seeds at each fixed epoch. Five seeds provide very coarse coverage evidence.
- A 24-hour arc at 800 km can leave drag unidentifiable. Broad uncertainty is a legitimate result.
- Fixed spacecraft properties allow a density interpretation in controlled tests. With mismatched force models, report an effective drag correction.
- Incomplete jobs, unconverged fits, invalid covariances, and failed NUTS diagnostics are visible in the manifests and HTML report. They are not successful results.